# Tool Use GRPO - Qwen2.5-0.5B (Small Model Baseline)

**Tinker RL Project — PES University MTech Capstone (Group 6)**

| Field | Value |
|-------|-------|
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` |
| **Benchmark** | Function Calling / Tool Use (glaive-function-calling-v2) |
| **Method** | GRPO + LoRA rank 32 |
| **Reward** | Binary: 1.0 if correct tool + args, 0.0 otherwise |
| **Training API** | Tinker (cloud GPU) |
| **Environment** | Atropos + custom ToolUseEnv |
| **Assigned to** | Arumugam |
| **Purpose** | Capacity threshold test — does a 0.5B model learn tool use at all? |
| **Status** | Ready to run |

**Hypothesis:** Based on the GSM8K capacity-threshold finding (Llama-3.2-3B failed; 8B succeeded),
we expect a 0.5B model to struggle. This experiment tests whether function calling requires a minimum
model capacity similar to mathematical reasoning.

In [ ]:
# Install dependencies
!pip install -q atroposlib==0.3.0 \
    'git+https://github.com/thinking-machines-lab/tinker.git' \
    datasets transformers wandb pydantic python-dotenv

In [ ]:
# Clone repo
!git clone https://github.com/pes-llm-research/tinker-rl-lab.git
%cd tinker-rl-lab/atropos
!pip install -e . -q

In [ ]:
import os

# Set credentials (use Colab secrets or paste directly)
os.environ['TINKER_API_KEY'] = 'YOUR_TINKER_API_KEY'
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['TINKER_CONFIG_PATH'] = 'configs/tool_use_qwen_0_5b.yaml'

print('Credentials set.')

In [ ]:
# Write config (already in repo; this cell verifies it)
import yaml
with open('configs/tool_use_qwen_0_5b.yaml') as f:
    cfg = yaml.safe_load(f)
print(f"Model: {cfg['openai'][0]['model_name']}")
print(f"Steps: {cfg['env']['total_steps']}")
print(f"Batch: {cfg['env']['batch_size']}")
print(f"LoRA rank: {cfg['tinker']['lora_rank']}")

In [ ]:
# Launch Atropos coordinator (background)
!nohup python -m atroposlib.server > /tmp/atropos_server.log 2>&1 &
import time; time.sleep(5)
print('Atropos coordinator started')

In [ ]:
# Launch environment (background)
!nohup python -m tinker_atropos.environments.tool_use_tinker \
    --config configs/tool_use_qwen_0_5b.yaml \
    > /tmp/tool_use_0_5b_env.log 2>&1 &
import time; time.sleep(10)
print('Environment started')
!tail -5 /tmp/tool_use_0_5b_env.log

In [ ]:
# Launch trainer (foreground — this runs the GRPO training loop)
!python launch_training.py --config configs/tool_use_qwen_0_5b.yaml

In [ ]:
# RESULTS — paste step data here after training completes
# Format: steps = [...], rewards = [...]

steps = []   # TODO: fill in after training
rewards = [] # TODO: fill in after training

# Comparison: Qwen3-8B tool-use run (expected to succeed)
# baseline_8b = {'initial': 0.05, 'final': 0.85}  # expected

if steps and rewards:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(steps, rewards, color='#e74c3c', linewidth=2, label='Qwen2.5-0.5B (this run)')
    ax.fill_between(steps, rewards, alpha=0.15, color='#e74c3c')
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.4, label='50% accuracy threshold')
    ax.set_xlabel('Training Step')
    ax.set_ylabel('Tool Call Accuracy')
    ax.set_title('Tool Use GRPO - Qwen2.5-0.5B\nCapacity Threshold Test')
    ax.set_ylim(-0.05, 1.1)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f'Initial: {rewards[0]:.4f} | Final: {rewards[-1]:.4f}')
    if rewards[-1] < 0.1:
        print('Result: CAPACITY THRESHOLD FAILURE — model too small to learn tool use')
    elif rewards[-1] > 0.5:
        print('Result: SUCCESS — 0.5B can learn tool use')
    else:
        print('Result: PARTIAL — marginal learning, inconclusive')
else:
    print('Run training first, then paste step/reward data here.')